# **Organelle Signature Analysis**

# <mark> TO DO:
- <mark> update batch process org morph to output quant_path / f"{dataset_name}_organelle_morphology_metrics.csv"
- <mark> need to update batch_process_interactions_quant() function to include consecutive else: statements (they should all be ... = existing_morpho_key)
- <mark> need to make sure get_org_morphology (and any similar functions can take in list_region_names and list_region_segs as None)

***Prior to this notebook, you should have already run through [2.0_quantification_setup](2.0_quantification_setup.ipynb) and have segmentation outputs of <ins>at least two</ins> organelles from the same cells.***


### 📍 **Purpose**
This notebook combines the methods explained in notebooks 2.1 through 2.4 and the `method_...` notebooks of `infer-subc` [Part 2 - Quantification](.) into a cohesive analysis pipeline we call the `Organelle Signature Analysis`. Organelle signature analysis was designed to quantitatively assess the complex multivarite phenotypes in cells visible with confocal microscopy techniques. It includes analysis of organelle morphology, inter-organellar interactions, organelle and interaction distribution, and cell region quantification. Our group has highlighted the use of this technique in a recent publication: ["Neurons and astrocytes have distinct organelle signatures and responses to stress." Cell Reports 2025](https://pubmed.ncbi.nlm.nih.gov/40956669/).

The `Organelle Signature Analysis` pipeline includes two consecutive steps:
1. 🧪 Batch process the quantitative analysis of *multiple images* from <ins>INDIVIDUAL EXPERIMENTS</ins>
2. 🧮 Summarize quantitative metrics *per image* across <INS>ONE OR MORE EXPERIMENTS</ins>

Below you will find an `explanation` the two steps listed above. Then, more succinct code blocks are included to `execute quantification` on your own data.

-----

## 🗂️ **Table of Contents**
The following sections are included in this notebook:

**IMPORTS AND LOAD IMAGE**

**EXPLANATION OF STEPS** - This section serves as *expository examples* of the functions used to quantify, batch process, and summarize organelle morphology.

🧪 **Batch process *multiple images* from a <ins>SINGLE EXPERIMENT</ins>**
- **`DEFINE`** - The batch_process_quantification() function
- **`TEST`** - The batch_process_quantification() function on a subset of data

🧮 **Summarize metrics *per image* across <INS>ONE OR MORE EXPERIMENTS</ins>**
- **`DEFINE`** - The batch_process_summarystats() function
- **`TEST`** - The batch_process_summarystats() function on a subset of data

**EXECUTE QUANTIFICATION** - Once you understand how the functions work, this section can be used to quantify your data in a quick and easy way.
- **`STEP 1`:** 🧪 **Batch process *multiple images* from a <ins>SINGLE EXPERIMENT</ins>**
- **`STEP 2`:** 🧮 **Summarize metrics *per image* across <INS>ONE OR MORE EXPERIMENTS</ins>**

-----
---------------------
## **IMPORTS AND LOAD IMAGE**
Details about the functions included in this subsection are outlined in the [`2.0_quantification_setup`](2.0_quantification_setup.ipynb) notebook. Please visit that notebook first if you are confused about any of the code included here.

In [11]:
from typing import List, Union
from pathlib import Path
import os
import time
import warnings

import numpy as np
import pandas as pd

from infer_subc.core.img import *
from infer_subc.core.file_io import export_inferred_organelle
from infer_subc.utils.batch import list_image_files, find_segmentation_tiff_files
from infer_subc.core.file_io import read_czi_image, read_tiff_image
from infer_subc.quantification.batch import load_existing_keys_csv, append_atomic_csv
from infer_subc.quantification.morphology import get_org_morphology
from infer_subc.quantification.interactions import get_interaction_metrics
from infer_subc.quantification.regions import get_regions_morphology
# from infer_subc.quantification.distribution import ...

pd.set_option('display.max_columns', None)

#### &#x1F3C3; **Run code; no user input required**

#### &#x1F6D1; &#x270D; **User Input Required:**

Please specify the following information about your data: `raw_img_type`, `data_root_path`, `raw_data_path`, `seg_data_path`, and `quant_data_path`. Information about these inputs are specified in notebook [2.0_quantification_setup.ipynb](2.0_quantification_setup.ipynb).

Also specify the following:
- `dataset_name`: is a unique string identifier for the dataset being processed. It will be included as metadata in output tables and as part of the output files names. It will be used to identify if any data has already been collected for this dataset.

In [16]:
#### USER INPUT REQUIRED ###
raw_img_type = ".czi"
data_root_path = Path(os.path.expanduser("~")) / "Documents/Python_Scripts/Infer-subc"
raw_data_path = data_root_path / "raw_two"
seg_data_path = data_root_path / "out_two"
quant_data_path = data_root_path / "quant_two/org-sig-analysis-TEST"

dataset_name = "20251218_test_dataset"

#### &#x1F3C3; **Run code; no user input required**

In [17]:
# Create the output directory to save the segmentation outputs in.
if not Path.exists(quant_data_path):
    Path.mkdir(quant_data_path)
    print(f"making {quant_data_path}")

# Create a list of the file paths for each image in the input folder. Select test image path.
raw_img_file_list = list_image_files(raw_data_path,raw_img_type)
print(f"Found {len(raw_img_file_list)} '{raw_img_type}' files in {raw_data_path}:")
display(pd.DataFrame({"Image Name":raw_img_file_list}))
print("\nThese will be used to test the organelle signature quantification pipeline in the EXPLANATION OF STEPS section.")

Found 2 '.czi' files in C:\Users\Shannon\Documents\Python_Scripts\Infer-subc\raw_two:


,Image Name
0,C:\Users\Shannon\Documents\Python_Scripts\Infer-subc\raw_two\a24hrs-Ctrl_14_Unmixing.czi
1,C:\Users\Shannon\Documents\Python_Scripts\Infer-subc\raw_two\a48hrs-Ctrl + oleic acid_01_Unmixing.czi



These will be used to test the organelle signature quantification pipeline in the EXPLANATION OF STEPS section.


#### &#x1F6D1; &#x270D; **User Input Required:**

Specify the following information about the segmentation files: - `org_file_names`, `org_channels_ordered`, `regions_file_names`, `suffix_separator`, and `mask_name`. Information about these inputs are specified in notebook [2.0_quantification_setup.ipynb](2.0_quantification_setup.ipynb).

Also specify the following:
- `channel_axis`: the axis of the channels in the raw data image. It can be determined by loading an image from your dataset (e.g., img) and looking at the image dimensions (e.g., img.shape) to see which dimension has the number of expected channels listed.
- `use_scale`: whether to use the pixel/voxel scale for each image when calculating metrics; if False, all measurements will be in pixel/voxel units and the XYZ ratios are assumed to be 1:1:1. 

In [18]:
#### USER INPUT REQUIRED ###
org_file_names = ["lyso", "mito", "golgi", "perox", "ER", "LD"]
org_channels_ordered = [1, 2, 3, 4, 5, 6]
regions_file_names = ["cell", "nuc"]
suffix_separator = "-20230426_test"
mask_name = "cell"

channel_axis = 0
use_scale = True

------
-----
## **EXPLANATION OF STEPS** <a id='explanation'></a>

-----
### 🧪 **Batch process *multiple images* from a <ins>SINGLE EXPERIMENT</ins>**

#### **`DEFINE` - The batch_process_org_morph() function**

The following code combines the analysis methods from notebooks [2.1](2.1_organelle_morphology.ipynb), [2.2](2.2_organelle_interactions.ipynb), [2.3](2.3_organelle_distribution.ipynb), and [2.4](2.4_cell_region_morphology.ipynb) into a single batch process function.

**Function requirements:**
- `dataset name` - a unique identifier for this particular dataset and quantitative approach. Here, we define a dataset as any images of the same samples/sample preparations taken within the same microscopy session. A example may be "rep-1". You may also choose to include an indicator of the date (or other helpful information) in case the same data will be processed again with slightly different settings.
- `file paths` to the image data and output location. This should include a *single folder* of intensity images, a *single folder* containing the corresponding organelle and mask/region segmentation images, and a location to save the quantitative output produced by this function.
- `raw file information` such as the raw file image type and channel information (axis and channel indexes for each organelle)
- `segmentation names` - a list of names used when saving the organelle and region segmentations, so that the corresponding segmentation images can be matched to the raw data for each image in the raw file location.
- `other information` including the name of the mask, to use the pixel/voxel scale or not, and any additional text used when naming the segmentation files. All of this information is option if not desired.
- `analysis settings` - specific settings necessary for interactions and distribution analyses.

**Basic steps:**
1. Check for any existing quantification data in the output location for this dataset. Skip any cells already quantified
2. Collect the list of files in the raw image location. For each image, find and organize the corresponding segmentation files for quantitative analysis.
3. Calculate the specified analyses per image and export to .csv file directly after output is generated.


**Function implementation:**

This function can be utilized from infer-subc using:
```python
infer_subc.quantification.batch.batch_process_quantification()
```

The code below includes a prototype of the batch_process_quantification function with extended comments. It is then tested on the subset of data specified in the `IMPORTS...` section above. The same function from `infer-subc` is also implemented is the `EXECUTE` section below.

#### &#x1F3C3; **Run code; no user input required**

&#x1F453; **FYI:** This code block defines a prototypes of the `batch_process_quantification()` function.

## <mark> NEED TO MAKE SURE EACH FILE GETS TESTED FOR DATA WITHIN; IF HALF PROCESSED, IT WILL WRITE THE DATA TWICE

In [ ]:
def _batch_process_quantification(dataset_name: str,
                                raw_path: Union[Path,str],
                                seg_path: Union[Path,str],
                                quant_path: Union[Path, str], 
                                raw_file_type: str,
                                channel_axis: int,
                                organelle_names: List[str],
                                organelle_channels: Union[List[int], None]=None,
                                region_names: Union[List[str], None]=None,
                                mask_name: Union[str, None]=None,
                                use_scale:bool=True,
                                seg_suffix:Union[str, None]=None,

                                # morphology settings
                                include_org_morpho:bool=True,

                                # regions settings
                                include_regions:bool=True,

                                # interaction settings
                                include_interactions:bool=True,
                                int_splitter:str="X",
                                include_inter_morpho:bool=True,
                                include_inter_degrees:bool=True,
                                export_inter_degree_imgs:bool=True,
                                export_interaction_sites:bool=True,
                                include_inter_dist:bool=True,

                                # distribution settings
                                include_org_dist:bool=True, 
                                dist_centering_obj: Union[str, None]=None,
                                dist_num_bins: Union[int, None]=5,
                                dist_center_on: Union[bool, None]=False,
                                dist_keep_center_as_bin: Union[bool, None]=True,
                                dist_zernike_degrees: Union[int, None]=9,
):
    """
    Batch process interaction quantification for a single dataset (e.g., images collected on the same data). 
    Morphology, distribution, and degree of interaction metrics analysis are all optionally available. 
    Interaction site segmentations and degree of interaction images can also be exported.
    
    Parameters:
    -----------
    dataset_name : str
        A unique string identifier for the dataset being processed. It will be included as metadata in output tables and as 
        part of the output files names. It will be used to identify if any data has already been collected for this dataset.
    raw_path : Union[Path,str]
        Path or str to the folder that contains the raw image files.
    seg_path : Union[Path,str]
        Path or str to the folder that contains the segmentation tiff files.
    quant_path : Union[Path, str]
        Path or str to the folder that the output datatables will be saved to.
    raw_file_type : str
        File type of the raw images (e.g., "czi", "tiff")
    channel_axis : int
        Axis corresponding to the channels in the image data
    organelle_names : List[str]
        List of organelle names to analyze. These names should match the suffix on the organelle segmentation files
    organelle_channels : Union[List[int], None]
        List of intensity channel indices in the raw files corresponding to each organelle included in organelle_names.
        The order should match organelle_names. 
        If no intensity analysis is to be included, specify None here.
    region_names : Union[List[str], None]
        List of region names to analyze. Usually ['cell', 'nuc'] for cell mask and nucleus.
        If no regions are to be included, specify None here.
    mask_name : Union[str, None]
        Name of the mask to use for segmentation (if any). This name should be included in the regions_name variable.
        If None, the entire image will be quantified.
    use_scale : bool
        Whether to apply scaling to the quantitative data; scaled data will be in real world units (e.g., microns) rather than pixels/voxels
    seg_suffix : Union[str, None]
        Any additional text that is included in the segmentation tiff files between the file stem and the segmentation suffix, not including the initial "-"
    int_splitter: str
        Character used to separate organelles within the interaction site names
        Ex) "mitoXlyso" for mito-lyso interactions
        include_morpho : bool, default=True
        Whether to compute morphology metrics
    include_morpho : bool, default=True
        Whether to compute morphology metrics for each interaction site
    include_interaction_degrees : bool, default=True
        Whether to compute interaction degree analysis
    include_dist : bool, default=True
        Whether to compute distribution metrics
    dist_centering_obj : str or None, default=None
        Name of the region to use for centering distribution analysis
        This region should be included in the list_region_names and list_region_segs variables
        If not specified, the center of the mask, or entire image if no mask was specified, will be used as the centering object
    dist_num_bins : int or None, default=5
        Number of radial bins to create in the XY distribution analysis
    dist_center_on : bool or None, default=True
        Whether to start creation of the XY bins from the center of the centering object (True) or edge (False)
    dist_keep_center_as_bin : bool or None
        Whether to keep centering object as the first XY bin
    dist_zernike_degrees : int or None, default=9
        Zernike polynomial degree for shape analysis in the XY distribution analysis
        If None and include_dist=True, no Zernike features will be calculated
    export_inter_degree_imgs : bool
        Whether to export interaction degree images
    export_interaction_sites : bool
        Whether to export interaction site images (including interaction site objects across the entire image; not masked)

    Returns:
    --------
    None
        Saves output files to the specified quantification path
    """
    #####
    # start timing & count for number of images processed
    batch_start = time.time()
    count = 0

    #####
    # check (and update if necessary) format of file paths
    if isinstance(raw_path, str): raw_path = Path(raw_path)
    if isinstance(seg_path, str): seg_path = Path(seg_path)
    if isinstance(quant_path, str): quant_path = Path(quant_path)

    # create output directory if it does not already exist
    if not Path.exists(quant_path):
        Path.mkdir(quant_path)
        print(f"making {quant_path}")

    #####
    # define output file paths & check if any existing data is present for this dataset
    # define unique keys in the output file for checking existing data
    unique_keys = ['dataset', 'image_name']

    # check for pre-existing organelle morphology metrics
    if include_org_morpho:
        org_morpho_path = quant_path / f"{dataset_name}_organelle_morphology_metrics.csv"
        # load existing data from organelle morphology csv file if it exists
        existing_org_morpho_keys = load_existing_keys_csv(org_morpho_path, unique_keys)
    else:
        # if not including this analysis, create an empty set (to be joined with other existing keys later)
        existing_org_morpho_keys = set()

    # check for pre-existing region morphology metrics
    if include_regions:
        regions_path = quant_path / f"{dataset_name}_regions_morphology_metrics.csv"
        existing_region_keys = load_existing_keys_csv(regions_path, unique_keys)
    else:
        # if not including this analysis, set existing keys to reflect the keys found in the organelle morphology analysis checked above
        existing_region_keys = set()

    # interaction metrics
    if include_interactions:
        # check for pre-existing interaction morphology metrics
        if include_inter_morpho:
            inter_morpho_tab_path = quant_path / f"{dataset_name}_interactions_morphology_metrics.csv"
            existing_inter_morpho_keys = load_existing_keys_csv(inter_morpho_tab_path, unique_keys)
        else:
            existing_inter_morpho_keys = set()
        
        # if interaction morphology not included, check for interaction labels file instead
        if not include_inter_morpho:
            inter_labs_tab_path = quant_path / f"{dataset_name}_interactions_labels.csv"
            existing_inter_labs_keys = load_existing_keys_csv(inter_labs_tab_path, unique_keys)
        else:
            existing_inter_labs_keys = set()

        # check for pre-existing interaction degree metrics
        if include_inter_degrees:
            int_degree_tab_path = quant_path / f"{dataset_name}_interactions_degree_metrics.csv"
            existing_int_degree_keys = load_existing_keys_csv(int_degree_tab_path, unique_keys)
        else:
            existing_int_degree_keys = set()

        # check for pre-existing interaction distribution metrics
        if include_inter_dist:
            inter_dist_tab_path = quant_path / f"{dataset_name}_interactions_distribution_metrics.csv"
            existing_inter_dist_keys = load_existing_keys_csv(inter_dist_tab_path, unique_keys)
        else:
            existing_inter_dist_keys = set()
        
        # define paths for exporting interaction degree images and interaction site segmentations if specified
        int_degree_img_path = quant_path / "interaction_degree_images" if export_inter_degree_imgs else None
        interaction_sites_path = quant_path / "interaction_site_segmentations" if export_interaction_sites else None

    else:
        # if interactions analysis is not included, set existing keys to reflect only the included analyses already checked
        existing_inter_morpho_keys = set()
        existing_int_degree_keys = set()
        existing_inter_dist_keys = set()

    if include_org_dist:
        org_dist_tab_path = quant_path / f"{dataset_name}_organelle_distribution_metrics.csv"
        existing_org_dist_keys = load_existing_keys_csv(org_dist_tab_path, unique_keys)
    else:
        existing_org_dist_keys = set()

    # define the set of existing keys across ALL included analyses
    interactions_existing_keys = existing_inter_morpho_keys.intersection(existing_inter_labs_keys).intersection(existing_int_degree_keys).intersection(existing_inter_dist_keys)
    existing_keys = existing_org_morpho_keys.intersection(existing_region_keys).intersection(existing_inter_morpho_keys).intersection(existing_int_degree_keys).intersection(existing_inter_dist_keys).intersection(existing_org_dist_keys)


    #####
    # create a combined list segmentation images to collect for each image
    # if no masks are provided, only organelle segmentations will be collected
    segs_to_collect = organelle_names + region_names if region_names is not None else organelle_names

    # list files from the raw path that will be processed below
    img_file_list = list_image_files(raw_path, raw_file_type)

    # print warning and stop if no files found
    len_file_list = len(img_file_list)
    if len_file_list==0:
        raise ValueError(f"No '{raw_file_type}' files found in {raw_path}. Please check the input path and file type.")

    # loop through list of images; analyze each and save the data the csv file
    for img_f in img_file_list:
        # start second timer for each image
        img_start = time.time()
        # increment count of images processed
        count = count+1

        # skip files that have already been processed based on existing keys
        if (dataset_name, img_f.stem) in existing_keys:
            print(f"Skipping {img_f.name} as it is already listed in the output file(s).")
            continue

        # if not in existing keys, process analysis for this cell
        else:
            # find file paths for raw and all segmentation files
            filez = find_segmentation_tiff_files(img_f, segs_to_collect, seg_path, seg_suffix)

            # read in raw image and metadata
            img_data, meta_dict = read_czi_image(filez["raw"])

            # format intensity channel separately from the raw file based on provided organelle channels input information
            # if not organelle channels provided, set to None (no intensity analysis will be performed)
            if organelle_channels is None:
                intensities = None
                print("No organelle channels provided; intensity analysis will be skipped.")
            else:
                if channel_axis != 0:
                    img_data = np.moveaxis(img_data, channel_axis, 0)
                intensities = [img_data[ch] for ch in organelle_channels]

            # store organelle segmentation images as list
            if organelle_names is None:
                raise ValueError("No organelle names provided; organelle segmentations are required for quantification.")
            else:
                organelles = [read_tiff_image(filez[org]) for org in organelle_names]

            # store regions segmentations as a list
            if region_names is None:
                regions = None
                mask_name = None  # if no regions provided, no mask can be applied
                print("No region names provided; regions analysis will not be calculated & no mask will be applied.")
            else:
                regions = [read_tiff_image(filez[r]) for r in region_names]

            # define the scale for quantification
            if use_scale is True:
                scale = meta_dict['scale']
                print(f"Using pixel/voxel scale: {scale}")
            else:
                scale = None
                print(f"No scale was provided; all measurements will be in pixel/voxel units.")

            # process organelle morphology analysis, if specified
            if include_org_morpho:
                # skip files that have already been processed for this analysis
                if (dataset_name, img_f.stem) in existing_org_morpho_keys:
                    print(f"Skipping organelle morphology analysis for {img_f.name} as it is already listed in the output file(s).")
                    continue
                else:
                    # get organelle morphology metrics for all included organelles
                    org_morph_metrics = get_org_morphology(source_file_path=img_f,
                                                    list_obj_names=organelle_names,
                                                    list_obj_segs=organelles,
                                                    list_intensity_img=intensities, 
                                                    list_region_names=region_names,
                                                    list_region_segs=regions, 
                                                    mask_name=mask_name,
                                                    scale=scale)
                
                    # add dataset name column to the organelle morphology data
                    org_morph_metrics.insert(loc=0,column='dataset',value=dataset_name)

                    # save the organelle morphology data for this image directly to csv
                    append_atomic_csv(org_morpho_path, org_morph_metrics)
                    del org_morph_metrics  # delete table variable to free up memory

            # process regions morphology analysis, if specified
            if include_regions and (region_names is not None):
                # skip files that have already been processed for this analysis
                if (dataset_name, img_f.stem) in existing_region_keys:
                    print(f"Skipping regions analysis for {img_f.name} as it is already listed in the output file(s).")
                    continue
                else:
                    # get region morphology metrics for all included regions
                    # intensity analysis will only be included for channels that correspond to organelles (as specified in the organelle_channels input)
                    regions_metrics = get_regions_morphology(source_file_path=img_f,
                                                        list_region_names=region_names,
                                                        list_region_segs=regions,
                                                        list_intensity_img=intensities,
                                                        list_channel_names=organelle_names,
                                                        mask_name=mask_name,
                                                        scale=scale)
                    
                    # add dataset name to regions table and save table directly to csv
                    regions_metrics.insert(loc=0,column='dataset',value=dataset_name)
                    append_atomic_csv(regions_path, regions_metrics)
                    del regions_metrics  # delete table to free up memory

            # process organelle interactions analysis, if specified
            # the same distribution analysis settings used for organelle distribution analysis will be applied to interaction site distribution analysis, if interactions distribution anlaysis included
            if include_interactions:
                # skip files that have already been processed for this analysis
                if (dataset_name, img_f.stem) in interactions_existing_keys:
                    print(f"Skipping {img_f.name} as it is already listed in the output file(s).")
                    continue
                else:
                    inter_dict, inter_morph_tab, inter_dist_tab, int_degree_img, int_degree_tab = get_interaction_metrics(source_file_path=img_f,
                                                                                                                        list_obj_names=organelle_names,
                                                                                                                        list_obj_segs=organelles,
                                                                                                                        list_intensity_img=intensities,
                                                                                                                        list_region_names=region_names,
                                                                                                                        list_region_segs=regions,
                                                                                                                        mask_name=mask_name,
                                                                                                                        splitter=int_splitter,
                                                                                                                        scale=scale,
                                                                                                                        include_morpho=include_inter_morpho,
                                                                                                                        include_interaction_degrees=include_inter_degrees,
                                                                                                                        include_dist=include_inter_dist, 
                                                                                                                        dist_centering_obj=dist_centering_obj,
                                                                                                                        dist_num_bins=dist_num_bins,
                                                                                                                        dist_center_on=dist_center_on,
                                                                                                                        dist_keep_center_as_bin=dist_keep_center_as_bin,
                                                                                                                        dist_zernike_degrees=dist_zernike_degrees)

                # if interactions morphology included, skip files that have already been processed for this analysis
                if include_inter_morpho:
                    if (dataset_name, img_f.stem) in interactions_existing_keys:
                        print(f"Skipping {img_f.name} as it is already listed in the output file(s).")
                        continue
                    else:
                        # save interaction morphology table data per image directly to csv
                        inter_morph_tab.insert(loc=0,column='dataset',value=dataset_name)
                        append_atomic_csv(inter_morpho_tab_path, inter_morph_tab)
                        del inter_morph_tab  # free up memory
                # if interactions morphology, not included, interactions labels are; check for files then save if necessary
                else:
                    if (dataset_name, img_f.stem) in existing_inter_labs_keys:
                        print(f"Skipping {img_f.name} as it is already listed in the output file(s).")
                        continue
                    else:
                        # save interaction labels table data per image directly to csv
                        inter_morph_tab.insert(loc=0,column='dataset',value=dataset_name)
                        append_atomic_csv(inter_labs_tab_path, inter_morph_tab)
                        del inter_morph_tab  # free up memory

                # if specified, export interaction site segmentation images
                if export_interaction_sites:
                    inter_site_cnt=0
                    for inter_name, inter_img in inter_dict.items():
                        inter_site_cnt+=1
                        # check that the file does not already exist before exporting
                        if not (Path(interaction_sites_path)/f"{img_f.name}-{inter_name}.tiff").exists():
                            export_inferred_organelle(inter_img, f"{inter_name}", meta_dict, interaction_sites_path)
                        else:
                            if inter_site_cnt<=1:
                                warnings.warn(f"Some of the interaction site images already exist for {img_f.name} in {interaction_sites_path}. They will not be overwritten.", UserWarning)
                del inter_dict  # free up memory
            
                # save the distribution table data per image directly to csv
                if include_inter_dist:
                    # skip files that have already been processed for this analysis
                    if (dataset_name, img_f.stem) in existing_inter_dist_keys:
                        print(f"Skipping {img_f.name} as it is already listed in the output file(s).")
                        continue
                    else:
                        # TODO: remove .astype(str) once method distribution functions have been updated
                        inter_dist_tab = inter_dist_tab.astype(str)  # ensure all data is string to avoid dtype issues
                        inter_dist_tab.insert(loc=0,column='dataset',value=dataset_name)
                        append_atomic_csv(inter_dist_tab_path, inter_dist_tab)
                del inter_dist_tab  # free up memory

                # save the degree table data per image directly to csv
                if include_inter_degrees:
                    # skip files that have already been processed for this analysis
                    if (dataset_name, img_f.stem) in existing_int_degree_keys:
                        print(f"Skipping {img_f.name} as it is already listed in the output file(s).")
                        continue
                    else:
                        int_degree_tab.insert(loc=0,column='dataset',value=dataset_name)
                        append_atomic_csv(int_degree_tab_path, int_degree_tab)

                    # save the degree image
                    if export_inter_degree_imgs:
                        if not (Path(int_degree_img_path)/f"{img_f.name}-interactions_degree.tiff").exists():
                            export_inferred_organelle(int_degree_img, f"interactions_degree", meta_dict, int_degree_img_path)
                        else:
                            warnings.warn(f"The {img_f.name}-interactions_degree.tiff image already exists in {int_degree_img_path}. It will not be overwritten.")
                del int_degree_tab  # free up memory
                del int_degree_img  # free up memory

        # process organelle distribution analysis, if specified
        if include_org_dist:
            if (dataset_name, img_f.stem) in existing_org_dist_keys:
                print(f"Skipping organelle distribution analysis for {img_f.name} as it is already listed in the output file(s).")
                continue
            else:
                print("ORG DIST STUFF NOT YET BATCHED!!")
                # org_dist_tab = get_org_distribution(source_file_path=img_f,
                #                                     list_obj_names=organelle_names,
                #                                     list_obj_segs=organelles,
                #                                     list_region_names=region_names,
                #                                     list_region_segs=regions,
                #                                     mask_name=mask_name,
                #                                     scale=scale,
                #                                     dist_centering_obj=dist_centering_obj,
                #                                     dist_num_bins=dist_num_bins,
                #                                     dist_center_on=dist_center_on,
                #                                     dist_keep_center_as_bin=dist_keep_center_as_bin,
                #                                     dist_zernike_degrees=dist_zernike_degrees)
                
                # # save interaction morphology table data per image directly to csv
                # org_dist_tab.insert(loc=0,column='dataset',value=dataset_name)
                # append_atomic_csv(org_dist_tab_path, org_dist_tab)
                # del org_dist_tab  # free up memory
    
        # end timer for single image
        end2 = time.time()
        print(f"Completed quantification of {img_f.name} in {(end2-img_start)/60} mins.")
        print(f"{count}/{len_file_list} images have been processed.")

    # end timer for entire batch
    batch_end = time.time()
    print(f"Quantification for {count} files is COMPLETE in {(batch_end - batch_start)/60} minutes! Files saved to '{quant_path}'.")

#### **`TEST` - The batch_process_org_morph() function**

#### &#x1F3C3; **Run code; no user input required**

&#x1F453; **FYI:** This code block applies the function above to your test image. The settings specified above are applied here.

In [22]:
_batch_process_quantification(dataset_name=dataset_name,
                              seg_path = seg_data_path,
                              quant_path = quant_data_path, 
                              raw_path = raw_data_path, 
                              raw_file_type = raw_img_type,
                              channel_axis=channel_axis,
                              organelle_names = org_file_names,
                              organelle_channels = org_channels_ordered,
                              region_names = regions_file_names,
                              mask_name = mask_name,
                              use_scale = use_scale,
                              seg_suffix = suffix_separator,
                              include_org_morpho=True,
                              include_regions=True,
                              include_interactions=True,
                              int_splitter="X",
                              include_inter_morpho=True,
                              include_inter_degrees=True,
                              export_inter_degree_imgs=True,
                              export_interaction_sites=True,
                              include_inter_dist=True,
                              include_org_dist=True,
                              dist_centering_obj=None,
                              dist_num_bins=5,
                              dist_center_on=True,
                              dist_keep_center_as_bin=True,
                              dist_zernike_degrees=9)

# read output files from path and print for inspection
org_morpho_path = quant_data_path / f"{dataset_name}_organelle_morphology_metrics.csv"
org_morpho1 = pd.read_csv(org_morpho_path)
regions_path = quant_data_path / f"{dataset_name}_regions_morphology_metrics.csv"
regions1 = pd.read_csv(regions_path)
inter_morpho_tab_path = quant_data_path / f"{dataset_name}_interactions_morphology_metrics.csv"
inter_morpho1 = pd.read_csv(inter_morpho_tab_path)
int_degree_tab_path = quant_data_path / f"{dataset_name}_interactions_degree_metrics.csv"
inter_degree1 = pd.read_csv(int_degree_tab_path)
inter_dist_tab_path = quant_data_path / f"{dataset_name}_interactions_distribution_metrics.csv"
inter_dist1 = pd.read_csv(inter_dist_tab_path)
org_dist_tab_path = quant_data_path / f"{dataset_name}_organelle_distribution_metrics.csv"
org_dist1 = pd.read_csv(org_dist_tab_path)

Using pixel/voxel scale: (0.3891184878080979, 0.07987165184837317, 0.07987165184837318)
Quantifying organelle morphology from a24hrs-Ctrl_14_Unmixing.czi
Warning(s) suppressed while quantifying lyso. See 'method_morphology.ipynb' notebook for more details.
Warning(s) suppressed while quantifying mito. See 'method_morphology.ipynb' notebook for more details.
Warning(s) suppressed while quantifying golgi. See 'method_morphology.ipynb' notebook for more details.
Warning(s) suppressed while quantifying perox. See 'method_morphology.ipynb' notebook for more details.
Warning(s) suppressed while quantifying ER. See 'method_morphology.ipynb' notebook for more details.
Warning(s) suppressed while quantifying LD. See 'method_morphology.ipynb' notebook for more details.
Quantifying region morphology from a24hrs-Ctrl_14_Unmixing.czi


KeyboardInterrupt: 

-----
### 🧮 **Summarize metrics *per image/mask* across <INS>ONE OR MORE EXPERIMENTS</ins>**

#### **`STEP 1` - Get the orgnaelle morphology .csv files**

#### &#x1F6D1; &#x270D; **User Input Required:**

&#x1F453; **FYI:** Here, the data output from batch_process_org_morph() from one or more datasets is combined before summarization.

Please specify the list file paths to include in this analysis:
- `csv_path_list`: a list of file paths that include the output csv files from batch_process_org_morph() for one or more datasets. If there is more than one dataset in each location, they will all be read in here as long as each dataset has a unique name (as specified the by dataset_name variable).

In [ ]:
### USER INPUT REQUIRED ###
csv_path_list = [quant_data_path]

##### &#x1F3C3; **Run code; no user input required**

&#x1F453; **FYI:** These steps collect all of the organelle interactions .csv quantification files from the list of locations and combines them so that there is now one data table per analysis type included.

Here, we will just use the single data path specified as quant_data_path earlier in the notebook, but multiple paths can be specified is desired. This path will have a few datasets in it if the above was run through fully.

In [ ]:
###################
# Read in the csv files and combine them into one of each type
###################
# create empty list to hold the morphology tables from different experiments
org_tabs = []

# loop through all of the locations listed above and find the _org_morph files; append them to the list above
for loc in csv_path_list:

    # list all csv files in the location
    files_store = sorted(loc.glob("*.csv"))

    # find the unique datasets in this location based on the prefixes before "_org_morphology_metrics"
    prefixes = set(f.name.split("_org_morphology_metrics")[0] for f in files_store if "_org_morphology_metrics" in f.name)
    print(f"Found the following datasets in {loc}:", prefixes)
    for prefix in prefixes:
        # select only the files from this dataset
        files_subset = [f for f in files_store if f.name.startswith(prefix +"_org_morphology_metrics")]
        for file in files_subset:
            stem = file.stem
            if "_org_morph" in stem:
                test_orgs = pd.read_csv(file, index_col=0)
                org_tabs.append(test_orgs)

# combine the org_morph lists found above into one table
org_df = pd.concat(org_tabs,axis=0, join='outer').reset_index()

print("Morphology metrics from all datasets:")
org_df

#### **`STEP 2` - Summarize the mean, median, and standard deviation of each metric per image/mask region**

##### &#x1F3C3; **Run code; no user input required**

&#x1F453; **FYI:** The block of code below summarizes the metrics per image/mask. The following are summarized:
- The number, total volume, volume fraction (total org vol/mask volume), and total surface area of each organelle per image/mask
- The mean, median and standard deviation of the "SA_to_volume_ratio", "equivalent_diameter", "extent", "euler_number", "solidity", and "axis_major_length" metrics for each organelle per image/mask

In [ ]:
###################
# summary stat group
###################
group_by = ['dataset', 'image_name', 'mask_name', 'scale', 'object']
sharedcolumns = ["SA_to_volume_ratio", "equivalent_diameter", "extent", "euler_number", "solidity", "axis_major_length"]
ag_func_standard = ['mean', 'median', 'std']

###################
# summarize shared measurements between org_df and contacts_df
###################
tab1 = org_df[group_by + ['label']].groupby(group_by).agg(['count'])
tab1.rename(columns={'label': 'org'}, inplace=True)
tab2 = org_df[group_by + ['volume', 'surface_area']].groupby(group_by).agg(['sum'] + ag_func_standard)
tab3 = org_df[group_by + sharedcolumns].groupby(group_by).agg(ag_func_standard)
org_summary = pd.merge(tab1, tab2, 'outer', on=group_by)
org_summary = pd.merge(org_summary, tab3, 'outer', on=group_by)

# Get mask_name and corresponding volume column per group & calculate volume fraction
mask_names = org_df.groupby(group_by)['mask_name'].first()
mask_volume_data = org_df.groupby(group_by).first().apply(lambda row: row[f"{mask_names.loc[row.name]}_volume"], axis=1)
org_summary.insert(org_summary.columns.get_loc(('volume', 'sum')) + 1, ('volume', 'fraction'), org_summary[('volume', 'sum')]/mask_volume_data)

print("Organelle morphology summary statistics:")
org_summary

#### **`STEP 3` - Ensure all organelles included in the analysis are represented and fill NA values with 0 as needed**

##### &#x1F3C3; **Run code; no user input required**

&#x1F453; **FYI:** The block of code below adds any organelles that were missing from the analysis (no organelle objects in images) and represents their volumes and counts with 0. If any objects have a count of 1, the standard deviation value is changes to NA.

In [ ]:
###################
# fill gaps & NA values
###################
# Ensure all possible interactions are represented (if missing fill with NaN)
for ind in org_summary.index.droplevel(4).unique().to_list():
    for row in org_file_names:
        if ind+(row,) not in org_summary.index:
            org_summary.loc[ind+(row,)] = np.nan

# fill NA with 0 for specific columns
fill_dict = {('org', 'count'): 0, 
             ('volume', 'sum'): 0,
             ('surface_area', 'sum'): 0,
             ('volume', 'fraction'): 0}
org_summary = org_summary.fillna(value=fill_dict)

# if (org, count) is 1, set mean, median, and std to NaN
single_site_mask = org_summary[('org', 'count')] == 1
for col in sharedcolumns+['volume', 'surface_area']:
    org_summary.loc[single_site_mask, (col, 'std')] = np.nan

org_summary.sort_index(inplace=True)

print("Morphology summary of interaction sites per image (or mask region):")
display(org_summary.head(7))

#### **`STEP 4` - Unstack the organelle names and save file**

#### &#x1F6D1; &#x270D; **User Input Required:**

&#x1F453; **FYI:** In this step each table is reformated to "unstack" the organelle column into the rows. In the resulting table, each row corresponds to a single image. The table is then exported.

Please specify if the interactions degree analysis should be carried out:
- `out_prefix`: A string used to name the output files. For example, if "date" is specified as out_prefix, the output files would be named "date_org_morphology_summarystats.csv".

In [ ]:
### USER INPUT REQUIRED ###
out_prefix = "all_datasets_1"

##### &#x1F3C3; **Run code; no user input required**

&#x1F453; **FYI:** The block of code below the following action occur:
1. The "organelle" column was unstacked resulting in a dataframe where each column if a different summary statistic and row represents the values for each cell. 
2. For specific values, such as organelle count and total/mean/median volume, NA values were filled with 0.
3. The standard deviation, mean, and median values for the ER (forced to be only one object) were removed.
4. The file was saved.

In [ ]:
###################
# flatten datasheet and export
###################
# export before unstacking
if (Path(quant_data_path) / f"{out_prefix}_per_org_morphology_summarystats.csv").exists():
    raise FileExistsError(f"CAUTION: {out_prefix}_per_org_morphology_summarystats.csv already exists and will not be overwritten. Move the existing file, change the `out_prefix` or `quant_data_path` to continue without error.")
else:
    org_summary.to_csv(str(quant_data_path) + f"/{out_prefix}_per_org_morphology_summarystats.csv", mode='x')
    print(f"Exported per-organelle morphology summary statistics (before unstacking) to {quant_data_path}/{out_prefix}_per_org_morphology_summarystats.csv")

org_morph_final = org_summary.unstack(-1)
org_morph_final.columns = ["_".join((col_name[1], col_name[-1], col_name[0])) for col_name in org_morph_final.columns.to_flat_index()]
org_morph_final.columns = [col.replace('sum', 'total') for col in org_morph_final.columns]
org_morph_final.columns = [col.replace(col, 'mask_volume') if 'mask' in col else col for col in org_morph_final.columns]
org_morph_final = org_morph_final.loc[:, ~org_morph_final.columns.duplicated()]
org_morph_final.reset_index(inplace=True)
print("The morphology summary after unstacking:")
display(org_morph_final)

###################
# export summary sheets
###################
if (Path(quant_data_path) / f"{out_prefix}_organelle_morphology_summarystats.csv").exists():
    raise FileExistsError(f"CAUTION: {out_prefix}_organelle_morphology_summarystats.csv already exists and will not be overwritten. Move the existing file, change the `out_prefix` or `quant_data_path` to continue without error.")
else:
    org_morph_final.to_csv(str(quant_data_path) + f"/{out_prefix}_organelle_morphology_summarystats.csv", mode='x')
    print(f"Exported organelle morphology summary statistics (after unstacking) to {quant_data_path}/{out_prefix}_organelle_morphology_summarystats.csv")

#### **`DEFINE` - The batch_org_morph_summary_stats() function**

In [ ]:
def _batch_org_morph_summary_stats(csv_path_list: List[str],
                                   out_path: str,
                                   out_prefix: str,
                                   organelle_names: List[str]):
    """" 
    csv_path_list: List[str],
        A list of path strings where .csv files to analyze are located.
    out_path: str,
        A path string where the summary data file will be output to
    out_prefix: str
        The prefix used to name the output file. An "_" will be included between this prefix and the file suffix.
    organelle_names: List[str],
        A list of organelle names used in the organelle morphology quantification (batch_process_org_morph function)
    """
    # for keeping track of dataset and file numbers
    ds_count = 0
    fl_count = 0

    ###################
    # Read in the csv files and combine them into one of each type
    ###################
    # create empty list to hold the morphology tables from different experiments
    org_tabs = []

    # loop through all of the locations listed above and find the _org_morph files; append them to the list above
    for loc in csv_path_list:
        # list all csv files in the location
        files_store = sorted(loc.glob("*.csv"))

        # find the unique datasets in this location based on the prefixes before "_org_morphology_metrics"
        prefixes = set(f.name.split("_org_morphology_metrics")[0] for f in files_store if "_org_morphology_metrics" in f.name)
        for prefix in prefixes:
            ds_count += 1
            # select only the files from this dataset
            files_subset = [f for f in files_store if f.name.startswith(prefix +"_org_morphology_metrics")]
            for file in files_subset:
                fl_count += 1
                stem = file.stem
                if "_org_morph" in stem:
                    test_orgs = pd.read_csv(file, index_col=0)
                    org_tabs.append(test_orgs)

    # combine the org_morph lists found above into one table
    org_df = pd.concat(org_tabs,axis=0, join='outer').reset_index()

    print(f"Found {fl_count} files from {ds_count} dataset(s) across {len(csv_path_list)} location(s).")


    ###################
    # summary stat group
    ###################
    group_by = ['dataset', 'image_name', 'mask_name', 'scale', 'object']
    sharedcolumns = ["SA_to_volume_ratio", "equivalent_diameter", "extent", "euler_number", "solidity", "axis_major_length"]
    ag_func_standard = ['mean', 'median', 'std']

    ###################
    # summarize shared measurements between org_df and contacts_df
    ###################
    tab1 = org_df[group_by + ['label']].groupby(group_by).agg(['count'])
    tab1.rename(columns={'label': 'org'}, inplace=True)
    tab2 = org_df[group_by + ['volume', 'surface_area']].groupby(group_by).agg(['sum'] + ag_func_standard)
    tab3 = org_df[group_by + sharedcolumns].groupby(group_by).agg(ag_func_standard)
    org_summary = pd.merge(tab1, tab2, 'outer', on=group_by)
    org_summary = pd.merge(org_summary, tab3, 'outer', on=group_by)

    # Get mask_name and corresponding volume column per group & calculate volume fraction
    mask_names = org_df.groupby(group_by)['mask_name'].first()
    mask_volume_data = org_df.groupby(group_by).first().apply(lambda row: row[f"{mask_names.loc[row.name]}_volume"], axis=1)
    org_summary.insert(org_summary.columns.get_loc(('volume', 'sum')) + 1, ('volume', 'fraction'), org_summary[('volume', 'sum')]/mask_volume_data)

    ###################
    # fill gaps & NA values
    ###################
    # Ensure all possible interactions are represented (if missing fill with NaN)
    for ind in org_summary.index.droplevel(4).unique().to_list():
        for row in organelle_names:
            if ind+(row,) not in org_summary.index:
                org_summary.loc[ind+(row,)] = np.nan

    # fill NA with 0 for specific columns
    fill_dict = {('org', 'count'): 0, 
                ('volume', 'sum'): 0,
                ('surface_area', 'sum'): 0,
                ('volume', 'fraction'): 0}
    org_summary = org_summary.fillna(value=fill_dict)

    # if (org, count) is 1, set mean, median, and std to NaN
    single_site_mask = org_summary[('org', 'count')] == 1
    for col in sharedcolumns+['volume', 'surface_area']:
        org_summary.loc[single_site_mask, (col, 'std')] = np.nan

    org_summary.sort_index(inplace=True)

    ###################
    # flatten datasheet and export
    ###################
    # export before unstacking
    if (Path(out_path) / f"{out_prefix}_per_org_morphology_summarystats.csv").exists():
        raise FileExistsError(f"CAUTION: {out_prefix}_per_org_morphology_summarystats.csv already exists and will not be overwritten. Move the existing file, change the `out_prefix` or `quant_data_path` to continue without error.")
    else:
        org_summary.to_csv(str(out_path) + f"/{out_prefix}_per_org_morphology_summarystats.csv", mode='x')
        print(f"Exported per-organelle morphology summary statistics (before unstacking) to {quant_data_path}/{out_prefix}_per_org_morphology_summarystats.csv")

    org_morph_final = org_summary.unstack(-1)
    org_morph_final.columns = ["_".join((col_name[1], col_name[-1], col_name[0])) for col_name in org_morph_final.columns.to_flat_index()]
    org_morph_final.columns = [col.replace('sum', 'total') for col in org_morph_final.columns]
    org_morph_final.columns = [col.replace(col, 'mask_volume') if 'mask' in col else col for col in org_morph_final.columns]
    org_morph_final = org_morph_final.loc[:, ~org_morph_final.columns.duplicated()]
    org_morph_final.reset_index(inplace=True)

    ###################
    # export summary sheets
    ###################
    if (Path(out_path) / f"{out_prefix}_organelle_morphology_summarystats.csv").exists():
        raise FileExistsError(f"CAUTION: {out_prefix}_organelle_morphology_summarystats.csv already exists and will not be overwritten. Move the existing file, change the `out_prefix` or `quant_data_path` to continue without error.")
    else:
        org_morph_final.to_csv(str(out_path) + f"/{out_prefix}_organelle_morphology_summarystats.csv", mode='x')
        print(f"Exported organelle morphology summary statistics (after unstacking) to {out_path}/{out_prefix}_organelle_morphology_summarystats.csv")
    print(f"Organelle morphology summary is complete.")
    return org_summary

#### &#x1F3C3; **Run code; no user input required**

&#x1F453; **FYI:** This code block applies the function above to your test image. The settings specified above are applied here.

In [ ]:
# define new out_prefix for comparison
out_prefix2 = "all_datasets_1-comparison"

# test the function above
test_org_summary = _batch_org_morph_summary_stats(csv_path_list = csv_path_list,
                                                  out_path = quant_data_path,
                                                  out_prefix = out_prefix2,
                                                  organelle_names = org_file_names)

# compare the output files from both dataset runs to ensure they are identical
morpho_file1 = quant_data_path / f"{out_prefix}_organelle_morphology_summarystats.csv"
morpho_file2 = quant_data_path / f"{out_prefix2}_organelle_morphology_summarystats.csv"

# determine if files are identical except for the dataset name column
if morpho_file1.exists() and morpho_file2.exists():
    morpho_1 = pd.read_csv(morpho_file1)
    morpho_2 = pd.read_csv(morpho_file2)
    print("\nMorphology summarystats files identical:", morpho_1.equals(morpho_2))

##### &#x1F453; **FYI:** This function has been added to `infer_subc.quantification.morphology` and can be imported with the following:
> ```python
> from infer_subc.quantification.morphology import batch_org_morph_summary_stats
> ```

In [ ]:
from infer_subc.quantification.morphology import batch_org_morph_summary_stats

# define new out_prefix for comparison
out_prefix3 = "all_datasets_1-FINAL"

# test the function above
test_org_summary = _batch_org_morph_summary_stats(csv_path_list = csv_path_list,
                                                  out_path = quant_data_path,
                                                  out_prefix = out_prefix3,
                                                  organelle_names = org_file_names)

# compare the output files from both dataset runs to ensure they are identical
morpho_file3 = quant_data_path / f"{out_prefix3}_organelle_morphology_summarystats.csv"
morpho_file2 = quant_data_path / f"{out_prefix2}_organelle_morphology_summarystats.csv"

# determine if files are identical except for the dataset name column
if morpho_file3.exists() and morpho_file2.exists():
    morph_3 = pd.read_csv(morpho_file3)
    morph_2 = pd.read_csv(morpho_file2)
    print("\nMorphology summarystats files identical:", morph_3.equals(morph_2))

-----
-----

## **EXECUTE QUANTIFICATION** <a id='execute'></a>

### **`STEP 1` - 🧪 Batch process *multiple cells* from a <ins>SINGLE EXPERIMENT</ins>**

#### &#x1F6D1; &#x270D; **User Input Required:**

Please specify the following information about your data: 
- `out_file_name`: The prefix to use when naming the output datatable. Do not add a separator; "_" will be added between your prefix and the base name given in the function below.
- `seg_path`: Path or str to the folder that contains the segmentation tiff files
- `out_path`: Path or str to the folder that the output datatables will be saved to
- `raw_path`: Path or str to the folder that contains the raw image files
- `raw_file_type`: The file type of the raw data; ex - ".tiff", ".czi"
- `organelle_names`: A list of all organelle names that will be analyzed; the names should be the same as the suffix used to name each of the tiff segmentation files. Note: the intensity measurements collect per region (from get_region_morphology_3D function) will only be from channels associated to these organelles 
- `organelle_channels`: A list of channel indices associated to respective organelle staining in the raw image; the indices should listed in same order in which the respective segmentation name is listed in organelle_names
- `region_names`: A list of regions, or masks, to measure; the order should correlate to the order of the channels in the "masks" output segmentation file
- `mask`: The name of the region to use as the mask when measuring the organelles; this should be one of the names listed in regions list; usually this will be the "cell" mask
- `scale`: A tuple that contains the real world dimensions for each dimension in the image (Z, Y, X)
- `seg_suffix`: Any additional text that is included in the segmentation tiff files between the file stem and the segmentation suffix, not including the initial "-"

The defaults below utilize the user input from the `IMPORTS` section.

In [ ]:
out_file_name = "20241204_test"
seg_path = seg_data_path
out_path = quant_data_path
raw_path = raw_data_path
raw_file_type = raw_img_type
channel_axis = channel_axis
organelle_names = org_file_names
organelle_channels = org_channels_ordered
region_names = regions_file_names
mask_name = mask_name
scale = True
seg_suffix = suffix_separator

#### &#x1F3C3; **Run code; no user input required**
&#x1F453; **FYI:** This code block uses the inputs provided above to run the batch processing. The table that is saved to you files is also printed below for easy access.

In [ ]:
batch_org_morph_table = batch_process_org_morph(out_file_name,
                                                seg_path,
                                                out_path, 
                                                raw_path, 
                                                raw_file_type,
                                                channel_axis,
                                                organelle_names,
                                                organelle_channels,
                                                region_names,
                                                mask_name,
                                                scale,
                                                seg_suffix)

batch_org_morph_table

### **`STEP 2` - 🧮 Summarize metrics *per cell* across <INS>ONE OR MORE EXPERIMENTS</ins>**

#### &#x1F6D1; &#x270D; **User Input Required:**

Please specify the following information about your data: 
- `csv_path_list`: A list of path strings where .csv files to analyze are located.
- `out_path`: A path string where the summary data file will be output to
- `out_preffix`: The prefix used to name the output file. An "_" will be included between this prefix and the file suffix.

In [ ]:
csv_path_list = csv_path_list
out_path = quant_data_path
out_prefix = out_prefix
organelle_names = org_file_names

#### &#x1F3C3; **Run code; no user input required**
&#x1F453; **FYI:** This code block uses the inputs provided above to run the batch processing. The table that is saved to you files is also printed below for easy access.

In [ ]:
test_org_summary = batch_org_morph_summary_stats(csv_path_list = csv_path_list,
                                                  out_path = out_path,
                                                  out_prefix = out_prefix,
                                                  organelle_names = organelle_names)

test_org_summary

-----
-----
# 🎉 **CONGRATULATIONS!!**
### **You've successfully quantified organelle morphology using the modular `2.1._organelle_morphology` notebook.**

Continue on to other quantification notebooks as needed:
- [2.2 Organelle interactions]() (amounts, size, shape)
- [2.3 Subcellular distribution]() in XY and Z separately (of organelles and interaction sites)
- [2.4 Cell morphology]() (size, shape)